# CNN per la classificazione di CIFAR-10

## Obiettivo

In questo notebook viene costruita e addestrata una Convolutional Neural Network sullo stesso training set, validation set e test set utilizzati per la MLP.

La CNN sarà confrontata con la baseline MLP, che ha ottenuto:

- migliore validation accuracy: 0,4136;
- epoca della migliore validation accuracy: 43;
- minima validation loss: 1,7127;
- epoca della minima validation loss: 41;
- parametri: 1.707.274.

Il test set non verrà utilizzato durante la scelta degli iperparametri.

In [1]:
import os
import random
import time
import json

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers


SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


(x_train_full, y_train_full), (x_test, y_test) = (
    keras.datasets.cifar10.load_data()
)

y_train_full = y_train_full.ravel()
y_test = y_test.ravel()

x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full,
    y_train_full,
    test_size=0.10,
    random_state=SEED,
    stratify=y_train_full
)

print("Training:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

c:\Users\Msi\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Training: (45000, 32, 32, 3) (45000,)
Validation: (5000, 32, 32, 3) (5000,)
Test: (10000, 32, 32, 3) (10000,)


In [3]:
def build_cnn():
    return keras.Sequential(
        [
            layers.Input(
                shape=(32, 32, 3),
                name="immagine_input"
            ),

            layers.Conv2D(
                filters=32,
                kernel_size=(3, 3),
                padding="same",
                activation="relu",
                name="conv_32_1"
            ),

            layers.Conv2D(
                filters=32,
                kernel_size=(3, 3),
                padding="same",
                activation="relu",
                name="conv_32_2"
            ),

            layers.MaxPooling2D(
                pool_size=(2, 2),
                name="pool_1"
            ),

            layers.Dropout(
                rate=0.25,
                name="dropout_25_1"
            ),

            layers.Conv2D(
                filters=64,
                kernel_size=(3, 3),
                padding="same",
                activation="relu",
                name="conv_64_1"
            ),

            layers.Conv2D(
                filters=64,
                kernel_size=(3, 3),
                padding="same",
                activation="relu",
                name="conv_64_2"
            ),

            layers.MaxPooling2D(
                pool_size=(2, 2),
                name="pool_2"
            ),

            layers.Dropout(
                rate=0.25,
                name="dropout_25_2"
            ),

            layers.Flatten(
                name="flatten"
            ),

            layers.Dense(
                units=128,
                activation="relu",
                name="dense_128"
            ),

            layers.Dropout(
                rate=0.50,
                name="dropout_50"
            ),

            layers.Dense(
                units=10,
                activation="softmax",
                name="output"
            )
        ],
        name="cnn_cifar10"
    )


cnn = build_cnn()
cnn.summary()

Model: "cnn_cifar10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_32_1 (Conv2D)              │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_32_2 (Conv2D)              │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling2D)           │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_25_1 (Dropout)          │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_64_1 (Conv2D)              │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_64_2 (Conv2D)              │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling2D)           │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_25_2 (Dropout)          │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_128 (Dense)               │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_50 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 591,274 (2.26 MB)

 Trainable params: 591,274 (2.26 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("cnn compilata ")

cnn compilata 


In [5]:
os.makedirs(r"C:\Users\Msi\Desktop\machine-learing\progetto-cifar-10\models", exist_ok=True)
os.makedirs(r"C:\Users\Msi\Desktop\machine-learing\progetto-cifar-10\results\figures", exist_ok=True)
os.makedirs(r"C:\Users\Msi\Desktop\machine-learing\progetto-cifar-10\results\metrics", exist_ok=True)

early_stopping_cnn = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    min_delta=0.001,
    restore_best_weights=True,
    verbose=1
)

checkpoint_cnn = keras.callbacks.ModelCheckpoint(
    filepath=r"C:\Users\Msi\Desktop\machine-learing\progetto-cifar-10\models\best_cnn.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

reduce_lr_cnn = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [6]:
start_time_cnn = time.perf_counter()

history_cnn = cnn.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=60,
    batch_size=64,
    callbacks=[
        early_stopping_cnn,
        checkpoint_cnn,
        reduce_lr_cnn
    ],
    verbose=2
)

training_time_cnn = (
    time.perf_counter() - start_time_cnn
)

print(
    f"Tempo totale CNN: "
    f"{training_time_cnn:.2f} secondi"
)

Epoch 1/60

Epoch 1: val_loss improved from None to 1.22633, saving model to C:\Users\Msi\Desktop\machine-learing\progetto-cifar-10\models\best_cnn.keras

Epoch 1: finished saving model to C:\Users\Msi\Desktop\machine-learing\progetto-cifar-10\models\best_cnn.keras
704/704 - 21s - 29ms/step - accuracy: 0.3832 - loss: 1.6880 - val_accuracy: 0.5592 - val_loss: 1.2263 - learning_rate: 0.0010
Epoch 2/60

Epoch 2: val_loss improved from 1.22633 to 1.10842, saving model to C:\Users\Msi\Desktop\machine-learing\progetto-cifar-10\models\best_cnn.keras

Epoch 2: finished saving model to C:\Users\Msi\Desktop\machine-learing\progetto-cifar-10\models\best_cnn.keras
704/704 - 19s - 27ms/step - accuracy: 0.5368 - loss: 1.2959 - val_accuracy: 0.6034 - val_loss: 1.1084 - learning_rate: 0.0010
Epoch 3/60

Epoch 3: val_loss improved from 1.10842 to 0.95974, saving model to C:\Users\Msi\Desktop\machine-learing\progetto-cifar-10\models\best_cnn.keras

Epoch 3: finished saving model to C:\Users\Msi\Desktop\

In [7]:
print(type(history_cnn))
print(history_cnn.history.keys())

<class 'keras.src.callbacks.history.History'>
dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss', 'learning_rate'])


In [8]:
import numpy as np

train_accuracy_cnn=np.array(
    history_cnn.history["accuracy"]
)

val_accuracy_cnn = np.array(history_cnn.history["val_accuracy"])

train_loss_cnn = np.array(history_cnn.history["loss"])

val_loss_cnn=np.array(history_cnn.history["val_loss"])

epochs_ran_cnn=len(train_loss_cnn)
print("epoche eseguite", epochs_ran_cnn)

epoche eseguite 34


In [9]:
best_accuracy_index_cnn=np.argmax(val_accuracy_cnn)

best_accuracy_epoch_cnn = (best_accuracy_index_cnn +1 )

best_val_accuracy_cnn = val_accuracy_cnn[best_accuracy_index_cnn]

print("il migliore validation accuracy. ", f"{best_val_accuracy_cnn:.4f}")
print("epoca della migliore validation accuracy", best_accuracy_epoch_cnn)

il migliore validation accuracy.  0.8020
epoca della migliore validation accuracy 32


In [10]:
best_loss_index_cnn = np.argmin(val_loss_cnn)

best_loss_epoch_cnn= (best_loss_index_cnn +1 )

minmum_val_loss_cnn= val_loss_cnn[best_loss_index_cnn]

print("minore validation loss:", f"{minmum_val_loss_cnn:.4f}")

print("epoca della minore vaditation loss",best_loss_epoch_cnn)

minore validation loss: 0.6189
epoca della minore vaditation loss 27


In [11]:
train_accuracy_at_best_loss_cnn = ( train_accuracy_cnn[best_loss_index_cnn])

val_accuracy_at_best_loss_cnn=(val_accuracy_cnn[best_loss_index_cnn])

train_loss_at_best_loss_cnn = (train_loss_cnn[best_loss_index_cnn])

gap_at_best_loss_cnn = (train_accuracy_at_best_loss_cnn - val_accuracy_at_best_loss_cnn)

print(f"training accuracy all'epoca" f"{best_loss_epoch_cnn}: " f"{val_accuracy_at_best_loss_cnn:.4f}")

print(f"training loss all'epoca" f"{best_loss_epoch_cnn}: " f"{train_loss_at_best_loss_cnn:.4f}")
print("generazione gap: " f"{gap_at_best_loss_cnn:.4f}")

training accuracy all'epoca27: 0.8012
training loss all'epoca27: 0.4720
generazione gap: 0.0288


In [12]:
restore_best_weights=True

In [ ]:
restored_val_loss_cnn, restored_val_accuracy_cnn = (cnn.evaluate(x_val,y_val,verbose=0))

print("validation  loss del modello ripristinato:", f"{restored_val_loss_cnn:.4f}")

print("validation accuracy del modello ripristinato:", f"{restored_val_accuracy_cnn:.4f}")


validation  loss del modello ripristinato: 0.6189
validation accuracy del modello ripristinato: 0.8012


In [14]:
print("differenza tra mminima loss registrata " "e loss ripristinata", abs(minmum_val_loss_cnn -restored_val_accuracy_cnn))

differenza tra mminima loss registrata e loss ripristinata 0.1822795867919922


In [ ]:
import os

cnn_model_path = "C:\Users\Msi\Desktop\machine-learing\models\best_mlp.keras"

print("checkpoint cnn presente", os.path.exists(cnn_model_path))

